## Constructed-data method demonstration: CONSTRUCTED original null/selection/correlation examples

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are labelled separately as replays. French raw, extracted, and processed files are not bundled, and the required rights and permitted uses have not been established; this Notebook does not analyze French data.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 5 准备层：训练窗口配对筛选（不含交易）

本 Notebook 调用 `projects/project05/preparation/screening_layer.py` 的函数：在冻结的训练窗口里对 French 10 行业的 45 个配对做 Engle–Granger 协整筛选，报告测试次数、临界值、BH 校正与训练之后窗口里冻结 spread 的诊断，并复现三个反例（零假设下的假阳性、选最好一对的选择效应、相关≠协整）。

- 真实数据：French 10 行业价值加权月收益复利成的总回报指数代理（聚合、已修订、非时点、不可交易）；构造数据：第 17 章的模拟序列与独立随机游走。
- 这是准备层：没有 entry/exit、双腿成本、卖空融资与回测，筛选结果不是收益证据。本 Notebook 不写文件，须从第一个单元顺序运行；零假设与选择效应用较少的重复次数（CLI 用 200 次）。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p05_screening_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
spec = importlib.util.spec_from_file_location("p05", ROOT / "projects/project05/preparation/screening_layer.py")
p05 = importlib.util.module_from_spec(spec); spec.loader.exec_module(p05)


## 1. 冻结规则与数据

In [ ]:
rules = p05.load_rules()
print("French real_levels and historical industry screening SKIPPED: no redistribution/use rights established")


## 2. 训练窗口筛选：测试次数、临界值、BH、Bonferroni

In [ ]:
print("SKIPPED historical French screening/perturbation comparison; constructed examples follow")


In [ ]:
print("SKIPPED historical French screening/perturbation comparison; constructed examples follow")


## 3. 筛选只用训练段：改动之后的数据不改训练统计量

In [ ]:
print("SKIPPED historical French screening/perturbation comparison; constructed examples follow")


## 4. 零假设下的假阳性与“选最好一对”的选择效应（构造，少量重复）

In [ ]:
small = {**rules, "null_calibration": {**rules["null_calibration"], "reps": 30}, "selection_effect": {**rules["selection_effect"], "reps": 30}}
null = p05.null_calibration(small)
sel = p05.selection_effect(small)
print({k: round(v, 3) for k, v in null.items() if isinstance(v, float)})
print({k: round(v, 3) for k, v in sel.items() if isinstance(v, float)})
assert null["mean_rejections_at_alpha"] > 1.0 and sel["mean_test_pvalue_of_selected_pair"] > 0.2

## 5. 相关≠协整，hedge ratio≠资本权重

In [ ]:
cc = p05.correlation_is_not_cointegration()
print(pd.DataFrame(cc).T.round(4))
w = p05.hedge_ratio_is_not_a_capital_weight()
print({k: round(v, 4) for k, v in w.items()})
assert cc["common_trend_not_cointegrated"]["engle_granger_pvalue"] > 0.05 > cc["cointegrated"]["engle_granger_pvalue"]

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
ax.bar(["null mean rejections", "BH mean rejections"], [null["mean_rejections_at_alpha"],null["mean_bh_rejections"]])
ax.set(title="CONSTRUCTED independent random walks: multiple tests",ylabel="Mean rejected pairs")
fig.tight_layout(); fig.savefig(RUN_DIR / "screening_null_CONSTRUCTED.png",dpi=130); plt.show()


## 6. 自己试一试

1. 把 `rules["alpha"]` 想成 0.10，数一数两个窗口里“显著”的配对个数和零假设下的期望个数，说明为什么“显著个数多了”不是发现。
2. 在第 2 节再加一个训练窗口（比如训练到 1985-12），写下总检验次数与相应的 Bonferroni 阈值；这个窗口应该事前还是事后加？
3. 在第 5 节改 `hedge_ratio_is_not_a_capital_weight(beta=0.8, price_y=50, price_x=100)`，算资本权重与净美元暴露。